<a href="https://colab.research.google.com/github/gyeongee/Document-AI/blob/main/Document_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.api_core.client_options import ClientOptions
from google.cloud import documentai_v1

# project_id = "MY_PROJECT_ID"
# processor_id = "MY_PROCESSOR_ID"
# location = "MY_PROCESSOR_LOCATION"      # 예: "us", "eu", "asia-northeast3"
# file_path = "/path/to/local/pdf"

opts = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")
client = documentai_v1.DocumentProcessorServiceClient(client_options=opts)

full_processor_name = client.processor_path(project_id, location, processor_id)

# (선택) Processor 확인
processor = client.get_processor(
    request=documentai_v1.GetProcessorRequest(name=full_processor_name)
)
print(f"Processor Name: {processor.name}")

# 파일 읽기
with open(file_path, "rb") as f:
    content = f.read()

raw_document = documentai_v1.RawDocument(
    content=content,
    mime_type="application/pdf",
)

result = client.process_document(
    request=documentai_v1.ProcessRequest(name=processor.name, raw_document=raw_document)
)
document = result.document

# ---------------------------
# 유틸: text_anchor로 실제 문자열 뽑기
# ---------------------------
def get_text(doc: documentai_v1.Document, text_anchor) -> str:
    """Document.text + text_anchor(segments)로 실제 텍스트를 복원한다."""
    if not text_anchor or not text_anchor.text_segments:
        return ""
    parts = []
    for seg in text_anchor.text_segments:
        start = int(seg.start_index) if seg.start_index is not None else 0
        end = int(seg.end_index) if seg.end_index is not None else 0
        parts.append(doc.text[start:end])
    return "".join(parts)

# ---------------------------
# 유틸: bounding_poly 출력용(좌표/정규화 둘 다 대응)
# ---------------------------
def poly_to_str(bounding_poly) -> str:
    """
    bounding_poly.vertices 또는 bounding_poly.normalized_vertices를 문자열로 변환한다.
    - vertices: pixel 좌표
    - normalized_vertices: 0~1 정규화 좌표
    """
    if not bounding_poly:
        return ""
    if getattr(bounding_poly, "vertices", None):
        vs = bounding_poly.vertices
        return " | ".join([f"({v.x},{v.y})" for v in vs])
    if getattr(bounding_poly, "normalized_vertices", None):
        vs = bounding_poly.normalized_vertices
        return " | ".join([f"({v.x:.4f},{v.y:.4f})" for v in vs])
    return ""

# ============================================================
# 1) 좌표(bounding box): Token 단위 출력
# ============================================================
print("\n====================")
print("1) TOKENS + BBOX")
print("====================")

for pi, page in enumerate(document.pages, start=1):
    print(f"\n--- Page {pi} ---")
    # token은 매우 많을 수 있으니 필요하면 상위 N개만 출력하도록 제한 가능
    for ti, token in enumerate(page.tokens, start=1):
        token_text = get_text(document, token.layout.text_anchor).replace("\n", " ")
        bbox = poly_to_str(token.layout.bounding_poly)
        conf = getattr(token.layout, "confidence", None)
        # confidence는 버전에 따라 없을 수 있음
        conf_str = f"{conf:.3f}" if isinstance(conf, (int, float)) else "N/A"
        print(f"[Token {ti}] '{token_text}' | bbox: {bbox} | conf: {conf_str}")

# ============================================================
# 2) block / paragraph 구조 출력
# ============================================================
print("\n====================")
print("2) BLOCKS / PARAGRAPHS")
print("====================")

for pi, page in enumerate(document.pages, start=1):
    print(f"\n--- Page {pi} ---")

    # Blocks
    print("\n[Blocks]")
    for bi, block in enumerate(page.blocks, start=1):
        block_text = get_text(document, block.layout.text_anchor).strip().replace("\n", " ")
        bbox = poly_to_str(block.layout.bounding_poly)
        print(f"- Block {bi}: {block_text[:200]}{'...' if len(block_text) > 200 else ''}")
        print(f"  bbox: {bbox}")

    # Paragraphs
    print("\n[Paragraphs]")
    for gi, para in enumerate(page.paragraphs, start=1):
        para_text = get_text(document, para.layout.text_anchor).strip().replace("\n", " ")
        bbox = poly_to_str(para.layout.bounding_poly)
        print(f"- Paragraph {gi}: {para_text[:200]}{'...' if len(para_text) > 200 else ''}")
        print(f"  bbox: {bbox}")

# ============================================================
# 3) key-value entities 출력 (Extractor 계열에서 주로 존재)
# ============================================================
print("\n====================")
print("3) ENTITIES (key-value)")
print("====================")

if not document.entities:
    print("document.entities 가 비어 있다. (OCR Processor면 정상일 수 있다.)")
else:
    for ei, ent in enumerate(document.entities, start=1):
        etype = ent.type_  # 필드 타입 (예: invoice_id, supplier_name 등)
        mention = ent.mention_text or ""  # 추출된 값(텍스트)
        conf = ent.confidence if ent.confidence is not None else None
        conf_str = f"{conf:.3f}" if isinstance(conf, (int, float)) else "N/A"

        print(f"\n[Entity {ei}] type: {etype} | conf: {conf_str}")
        print(f"  value(mention_text): {mention}")

        # 엔티티가 문서 어디서 나왔는지(좌표) 보고 싶으면:
        # ent.page_anchor.page_refs[*].bounding_poly 를 확인
        if ent.page_anchor and ent.page_anchor.page_refs:
            for ref_i, ref in enumerate(ent.page_anchor.page_refs, start=1):
                page_num = ref.page if ref.page is not None else "N/A"
                bbox = poly_to_str(ref.bounding_poly)
                print(f"  - page_ref {ref_i}: page={page_num} bbox={bbox}")

Document AI 결과를 “내 스키마로 매핑”하기

In [ ]:
def entities_to_my_schema(document: documentai_v1.Document) -> dict:
    """
    document.entities 를 내가 원하는 스키마로 변환한다.
    - OCR Processor면 entities가 비어 있을 수 있다.
    """
    out = {
        "meta": {
            "pages": len(document.pages),
        },
        "fields": {}
    }

    # Document AI 엔티티가 존재할 때만
    for ent in (document.entities or []):
        key = ent.type_                      # 예: "invoice_id", "supplier_name"
        val = ent.mention_text or ""         # 추출된 값
        conf = float(ent.confidence or 0.0)

        # 동일 key가 여러 번 나올 수 있으므로 list로 적재
        out["fields"].setdefault(key, []).append({
            "value": val,
            "confidence": conf,
        })

    return out

# 사용
my_schema = entities_to_my_schema(document)
print(my_schema)

In [ ]:
# 원하는 키 이름으로 바꾸고 싶으면 매핑 테이블을 두기
KEY_MAP = {
    "invoice_id": "invoiceNumber",
    "supplier_name": "vendorName",
    "invoice_date": "issuedDate",
    "total_amount": "total",
}

def entities_to_custom_schema(document: documentai_v1.Document) -> dict:
    out = {"fields": {}}
    for ent in (document.entities or []):
        src_key = ent.type_
        dst_key = KEY_MAP.get(src_key, src_key)  # 매핑 없으면 원래 키 유지
        out["fields"].setdefault(dst_key, []).append(ent.mention_text or "")
    return out

(entities가 없을 때) “내가 직접 필드 추출 규칙을 추가”해서 스키마 구성

In [ ]:
import re

DATE_PATTERNS = [
    r"\b(20\d{2})[.\-/](0?[1-9]|1[0-2])[.\-/](0?[1-9]|[12]\d|3[01])\b",  # 2024-03-02 / 2024.3.2
    r"\b(20\d{2})년\s*(0?[1-9]|1[0-2])월\s*(0?[1-9]|[12]\d|3[01])일\b",  # 2024년 3월 2일
]

def extract_date_fallback(text: str) -> str | None:
    for pat in DATE_PATTERNS:
        m = re.search(pat, text)
        if m:
            return m.group(0)
    return None

def build_schema_with_fallback(document: documentai_v1.Document) -> dict:
    out = entities_to_my_schema(document)  # 먼저 entities 기반으로 구성
    text = document.text or ""
    if "invoice_date" not in out["fields"]:
      # 전체 텍스테어 날짜 패턴을 찾아 d에 넣음
        d = extract_date_fallback(text)
        if d:
          # 날짜 문자열을 찾았으면 out["fields"]["invoice_date"]를 강제 추가
          # "confidence": 0.0 -> 내가 정규식으로 추정해서 넣은 값이므로 신뢰도는 별도로 표기하지 못함
            out["fields"]["invoice_date"] = [{"value": d, "confidence": 0.0}]
    return out

In [ ]:
import re
from typing import Optional

# 날짜 패턴(필요하면 더 추가)
DATE_PATTERNS = [
    r"\b(20\d{2})[.\-/](0?[1-9]|1[0-2])[.\-/](0?[1-9]|[12]\d|3[01])\b",  # 2024-03-02 / 2024.3.2
    r"\b(20\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])\b",              # 20240302
    r"\b(20\d{2})\s*(0?[1-9]|1[0-2])\s*(0?[1-9]|[12]\d|3[01])\b",      # 2024 3 2 (희박하지만 가끔)
    r"\b(20\d{2})년\s*(0?[1-9]|1[0-2])월\s*(0?[1-9]|[12]\d|3[01])일\b", # 2024년 3월 2일
]

# 날짜 “의미” 키워드(문서 케이스에 맞게 확장 가능)
DATE_KEYWORDS = [
    "발행일", "작성일", "청구일", "거래일", "결제일", "발급일", "일자", "일시", "DATE",
    "Invoice Date", "Issue Date", "Issued Date", "Billing Date", "Transaction Date",
    "Date of Issue", "Document Date",
]

def _find_first_date_in_text(text: str) -> Optional[str]:
    """주어진 text 범위에서 첫 번째 날짜 매칭값을 반환한다."""
    for pat in DATE_PATTERNS:
        m = re.search(pat, text)
        if m:
            return m.group(0)
    return None

def extract_date_near_keywords(
    text: str,
    window: int = 80,         # 키워드 주변 탐색 범위(앞뒤 window 글자)
    max_hits: int = 50        # 키워드 매칭이 너무 많을 때 안전장치
) -> Optional[dict]:
    """
    문서 전체 text에서 '키워드 근처'의 날짜를 우선 추출한다.
    - 키워드가 발견되면 그 주변 window 글자 범위에서 날짜 패턴을 찾는다.
    - 여러 키워드/여러 위치가 있으면 가장 먼저 '날짜가 실제로 발견되는' 후보를 반환한다.
    반환 예:
      {"value": "2024-03-02", "source": "keyword_window", "keyword": "Invoice Date"}
    """
    if not text:
        return None

    # 키워드 길이가 다르고 대소문자 문제가 있어서, 영문은 대소문자 무시 검색을 같이 고려한다.
    # 여기서는 간단히: 키워드별로 re.escape 처리 후 검색한다.
    hits = []
    for kw in DATE_KEYWORDS:
        # 영문 키워드 포함 가능 → IGNORECASE 적용
        pattern = re.compile(re.escape(kw), flags=re.IGNORECASE)
        for m in pattern.finditer(text):
            hits.append((m.start(), m.end(), kw))
            if len(hits) >= max_hits:
                break
        if len(hits) >= max_hits:
            break

    # 키워드가 전혀 없으면 None (상위 로직에서 전체 텍스트 fallback로 넘어가게 설계)
    if not hits:
        return None

    # 키워드 등장 순서대로(문서 앞에서부터) 검사
    hits.sort(key=lambda x: x[0])

    for start, end, kw in hits:
        left = max(0, start - window)
        right = min(len(text), end + window)
        snippet = text[left:right]

        d = _find_first_date_in_text(snippet)
        if d:
            return {"value": d, "source": "keyword_window", "keyword": kw, "window": window}

    return None

def extract_date_fallback_smart(text: str) -> Optional[dict]:
    """
    1) 키워드 근처(window)에서 날짜 우선 추출
    2) 실패 시 전체 텍스트에서 날짜 추출
    반환 예:
      {"value": "...", "source": "keyword_window" or "regex_fallback", ...}
    """
    near = extract_date_near_keywords(text, window=80)
    if near:
        return near

    d = _find_first_date_in_text(text)
    if d:
        return {"value": d, "source": "regex_fallback"}
    return None

def build_schema_with_keyword_date_fallback(document) -> dict:
    """
    entities 기반 스키마(out)를 만든 뒤,
    invoice_date가 없으면 '키워드 근처 날짜'를 우선으로 보강한다.
    """
    out = entities_to_my_schema(document)  # 기존에 만든 함수(entities -> schema)
    text = document.text or ""

    # 이미 invoice_date가 있으면 건드리지 않는다.
    if "invoice_date" not in out.get("fields", {}):
        found = extract_date_fallback_smart(text)
        if found:
            out.setdefault("fields", {})
            out["fields"]["invoice_date"] = [{
                "value": found["value"],
                "confidence": 0.0,              # 정규식 기반 추정이므로 0.0
                "source": found.get("source"),  # "keyword_window" 또는 "regex_fallback"
                "keyword": found.get("keyword"),
                "window": found.get("window"),
            }]

    return out

In [ ]:
if entity.normalized_value:
    date = entity.normalized_value.date_value
else:
    date = regex_parse(entity.mention_text)